# Section 3.1: Duplicate Handling Pipeline

This notebook demonstrates a production-ready duplicate handling pipeline with comprehensive audit logging.

**Objective:** Identify and remove exact row duplicates and customer-level duplicates before feature engineering.

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple, Any
import logging

# Configure logging for audit trail
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

## PHASE 0: CONFIGURATION

Define all duplicate handling parameters in a single configuration dict.

In [ ]:
# ==============================================================================
# DUPLICATE HANDLING CONFIGURATION
# ==============================================================================

DUPLICATE_CONFIG = {
    'check_exact_rows': True,
    'check_customer_level': True,
    'customer_id_column': 'customer_id',
    'keep': 'first',  # Keep first occurrence, remove subsequent duplicates
    'fail_if_duplicates_remain': True,  # Raise error if duplicates still exist after cleaning
    'subset': None,  # None = check all columns; or specify list of columns to check
}

## PHASE 1: DEFINE DUPLICATE HANDLING FUNCTION

This function performs both exact-row and customer-level duplicate checks with detailed audit logging.

In [ ]:
def handle_duplicates(df: pd.DataFrame, config: Dict[str, Any] = None) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Detect and remove duplicate records at both exact-row and customer-level.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe to check for duplicates
    config : Dict
        Configuration dict with keys:
        - 'check_exact_rows': bool, whether to check for exact row duplicates
        - 'check_customer_level': bool, whether to check customer_id-level duplicates
        - 'customer_id_column': str, name of customer ID column
        - 'keep': str, which duplicate to keep ('first', 'last', False)
        - 'fail_if_duplicates_remain': bool, raise error if duplicates remain
        - 'subset': list or None, columns to consider for duplication
    
    Returns:
    --------
    df_clean : pd.DataFrame
        Dataframe with duplicates removed
    audit : Dict
        Audit trail with detailed logging of duplicate detection and removal
    """
    
    if config is None:
        config = DUPLICATE_CONFIG
    
    # Initialize audit trail
    audit = {
        'status': 'STARTED',
        'rows_input': len(df),
        'rows_removed': 0,
        'checks_performed': [],
        'duplicate_details': {},
        'errors': []
    }
    
    df_clean = df.copy()
    
    # =========================================================================
    # CHECK 1: EXACT ROW DUPLICATES
    # =========================================================================
    if config['check_exact_rows']:
        n_exact_dupes = df_clean.duplicated(subset=config['subset'], keep=False).sum()
        audit['checks_performed'].append('exact_row_check')
        
        if n_exact_dupes > 0:
            # Get the duplicate rows for inspection
            dup_mask = df_clean.duplicated(subset=config['subset'], keep=False)
            dup_rows = df_clean[dup_mask].sort_values(
                by=list(df_clean.columns[:5])
            ).reset_index(drop=True)
            
            audit['duplicate_details']['exact_duplicates'] = {
                'count': n_exact_dupes,
                'duplicate_pairs': len(df_clean[dup_mask]) // 2,  # Approximate
                'sample_indices': dup_rows.index.tolist()[:6]  # First 6 for inspection
            }
            
            logger.info(f"✓ Exact row check: Found {n_exact_dupes} duplicate rows")
            logger.info(f"  Sample of duplicates (first 6 rows):")
            logger.info(dup_rows.head(6))
        else:
            audit['duplicate_details']['exact_duplicates'] = {'count': 0}
            logger.info(f"✓ Exact row check: No exact duplicates found")
    
    # =========================================================================
    # CHECK 2: CUSTOMER-LEVEL DUPLICATES
    # =========================================================================
    if config['check_customer_level'] and config['customer_id_column'] in df_clean.columns:
        customer_col = config['customer_id_column']
        n_unique_customers = df_clean[customer_col].nunique()
        n_total_rows = len(df_clean)
        n_customer_dupes = n_total_rows - n_unique_customers
        
        audit['checks_performed'].append('customer_level_check')
        
        if n_customer_dupes > 0:
            # Identify which customers appear multiple times
            dup_customer_mask = df_clean[customer_col].duplicated(keep=False)
            dup_customers = df_clean[dup_customer_mask].sort_values(customer_col)
            
            # Get summary of duplicates per customer
            dupes_per_customer = df_clean[customer_col].value_counts()
            dupes_per_customer = dupes_per_customer[dupes_per_customer > 1]
            
            audit['duplicate_details']['customer_level'] = {
                'duplicate_customers': len(dupes_per_customer),
                'duplicate_rows': n_customer_dupes,
                'distribution': dupes_per_customer.to_dict(),
            }
            
            logger.info(f"✓ Customer-level check: {len(dupes_per_customer)} customers appear multiple times")
            logger.info(f"  Total rows involved in customer-level duplicates: {n_customer_dupes}")
            logger.info(f"  Distribution: {dict(dupes_per_customer)}")
            logger.info(f"\n  Sample of duplicated customers:")
            logger.info(dup_customers.head(8))
        else:
            audit['duplicate_details']['customer_level'] = {
                'duplicate_customers': 0,
                'duplicate_rows': 0
            }
            logger.info(f"✓ Customer-level check: No customer-level duplicates found")
    
    # =========================================================================
    # REMOVAL: DROP DUPLICATES
    # =========================================================================
    rows_before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=config['subset'], keep=config['keep'])
    rows_removed = rows_before - len(df_clean)
    
    audit['rows_removed'] = int(rows_removed)
    audit['rows_output'] = len(df_clean)
    
    if rows_removed > 0:
        logger.info(f"\n✓ Duplicate removal: {rows_removed} rows removed")
        logger.info(f"  Input rows: {rows_before} → Output rows: {len(df_clean)}")
    else:
        logger.info(f"\n✓ No duplicates to remove")
    
    # =========================================================================
    # VALIDATION: Confirm no duplicates remain
    # =========================================================================
    remaining_exact_dupes = df_clean.duplicated(subset=config['subset']).sum()
    
    if remaining_exact_dupes > 0:
        error_msg = f"Duplicate removal failed: {remaining_exact_dupes} duplicates still remain"
        audit['errors'].append(error_msg)
        audit['status'] = 'FAILED'
        logger.error(f"✗ VALIDATION FAILED: {error_msg}")
        
        if config['fail_if_duplicates_remain']:
            raise ValueError(error_msg)
    else:
        audit['status'] = 'SUCCESS'
        logger.info(f"\n✓ VALIDATION PASSED: No duplicates remain")
    
    logger.info(f"\n" + "="*70)
    logger.info(f"DUPLICATE HANDLING SUMMARY")
    logger.info(f"="*70)
    logger.info(f"Status: {audit['status']}")
    logger.info(f"Rows removed: {audit['rows_removed']}")
    logger.info(f"Input shape: ({audit['rows_input']}, {df.shape[1]})")
    logger.info(f"Output shape: ({audit['rows_output']}, {df_clean.shape[1]})")
    logger.info(f"="*70)
    
    return df_clean, audit

## PHASE 2: CREATE SAMPLE DATA FOR DEMONSTRATION

Generate a sample dataset with known duplicates to demonstrate the pipeline.

In [ ]:
# Create sample data with duplicates for demonstration
np.random.seed(42)

sample_data = pd.DataFrame({
    'customer_id': [1001, 1001, 1002, 1003, 1003, 1003, 1004, 1005, 1006, 1007],
    'age': [28, 28, 35, 42, 42, 42, 31, 29, 38, 45],
    'income': [45000, 45000, 62000, 78000, 78000, 78000, 51000, 48000, 69000, 95000],
    'credit_score': [720, 720, 680, 750, 750, 750, 710, 695, 740, 800],
    'loan_amount': [15000, 15000, 25000, 35000, 35000, 35000, 20000, 18000, 30000, 50000],
    'loan_approved': [1, 1, 1, 1, 1, 1, 0, 0, 1, 1]
})

print("SAMPLE DATA (with known duplicates):")
print(sample_data)
print(f"\nShape: {sample_data.shape}")
print(f"Unique customers: {sample_data['customer_id'].nunique()}")

## PHASE 3: RUN THE DUPLICATE HANDLING PIPELINE

In [ ]:
# Execute the duplicate handling pipeline
data_cleaned, audit_trail = handle_duplicates(sample_data, config=DUPLICATE_CONFIG)

## PHASE 4: DISPLAY AUDIT TRAIL

In [ ]:
# Print detailed audit trail
print("\n" + "="*70)
print("DETAILED AUDIT TRAIL")
print("="*70)
print(f"\nStatus: {audit_trail['status']}")
print(f"Input rows: {audit_trail['rows_input']}")
print(f"Output rows: {audit_trail['rows_output']}")
print(f"Rows removed: {audit_trail['rows_removed']}")
print(f"\nChecks performed: {', '.join(audit_trail['checks_performed'])}")

print(f"\nDuplicate Details:")
for check_type, details in audit_trail['duplicate_details'].items():
    print(f"\n  {check_type}:")
    for key, value in details.items():
        print(f"    {key}: {value}")

if audit_trail['errors']:
    print(f"\nErrors: {audit_trail['errors']}")
else:
    print(f"\nErrors: None")

## PHASE 5: INSPECT CLEANED DATA

In [ ]:
print("\nCLEANED DATA:")
print(data_cleaned)
print(f"\nShape: {data_cleaned.shape}")
print(f"Unique customers: {data_cleaned['customer_id'].nunique()}")

# Verify no duplicates remain
print(f"\nDuplicate check - remaining exact duplicates: {data_cleaned.duplicated().sum()}")
print(f"Duplicate check - remaining customer-level duplicates: {len(data_cleaned) - data_cleaned['customer_id'].nunique()}")

## PHASE 6: INTEGRATION INTO FULL PIPELINE (Template)

This is how you'll integrate duplicate handling into the full `DataCleaningPipeline` class.

In [ ]:
class DataCleaningPipeline:
    """
    Orchestrates the full data cleaning pipeline.
    Demonstrates production-ready, modular code structure.
    """
    
    def __init__(self, config: Dict[str, Any]):
        """
        Initialize the pipeline with configuration.
        
        Parameters:
        -----------
        config : Dict
            Configuration dict containing:
            - 'duplicate_handling': Config for handle_duplicates()
            - 'missing_value_handling': Config for handle_missing_values() [future]
            - 'outlier_handling': Config for handle_outliers() [future]
        """
        self.config = config
        self.audit_trail = {}
    
    def execute(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        """
        Run all cleaning steps in sequence.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Raw input dataframe
        
        Returns:
        --------
        df_clean : pd.DataFrame
            Cleaned dataframe
        audit_trail : Dict
            Comprehensive audit log of all cleaning steps
        """
        df_current = df.copy()
        
        # Step 1: Handle duplicates
        logger.info("\n" + "#"*70)
        logger.info("# STEP 1: DUPLICATE HANDLING")
        logger.info("#"*70)
        df_current, audit = handle_duplicates(df_current, self.config['duplicate_handling'])
        self.audit_trail['duplicates'] = audit
        
        # Step 2: Handle missing values [FUTURE]
        # logger.info("\n" + "#"*70)
        # logger.info("# STEP 2: MISSING VALUE HANDLING")
        # logger.info("#"*70)
        # df_current, audit = handle_missing_values(df_current, self.config['missing_value_handling'])
        # self.audit_trail['missing_values'] = audit
        
        # Step 3: Handle outliers [FUTURE]
        # logger.info("\n" + "#"*70)
        # logger.info("# STEP 3: OUTLIER HANDLING")
        # logger.info("#"*70)
        # df_current, audit = handle_outliers(df_current, self.config['outlier_handling'])
        # self.audit_trail['outliers'] = audit
        
        return df_current, self.audit_trail



## PHASE 7: EXECUTE FULL PIPELINE (Demo)

In [ ]:
# Full pipeline configuration
PIPELINE_CONFIG = {
    'duplicate_handling': DUPLICATE_CONFIG,
    # 'missing_value_handling': {...},  # Future
    # 'outlier_handling': {...},  # Future
}

# Initialize and execute pipeline
pipeline = DataCleaningPipeline(PIPELINE_CONFIG)
data_cleaned, full_audit_trail = pipeline.execute(sample_data)

## PHASE 8: FINAL PIPELINE SUMMARY

In [ ]:
# Display final summary
print("\n" + "="*70)
print("FINAL PIPELINE EXECUTION SUMMARY")
print("="*70)

for phase_name, phase_audit in full_audit_trail.items():
    print(f"\n{phase_name.upper()}:")
    print(f"  Status: {phase_audit['status']}")
    print(f"  Rows removed: {phase_audit['rows_removed']}")
    print(f"  Checks performed: {', '.join(phase_audit['checks_performed'])}")

print(f"\nOVERALL:")
print(f"  Input rows: {sample_data.shape[0]}")
print(f"  Output rows: {data_cleaned.shape[0]}")
print(f"  Total rows removed: {sample_data.shape[0] - data_cleaned.shape[0]}")
print(f"  Columns: {data_cleaned.shape[1]}")
print(f"\n  ✓ Data quality check PASSED")
print("="*70)